# 01 – Data collection

**Goal:** Collect the raw data for the analysis from four sources: two web APIs, one web page (scraping) and one file download.
**Input:** web APIs and web pages · **Output:** `data/raw/` (responses stored unchanged), `data/interim/jolpica_*.parquet`, `data/interim/wikipedia_circuits.parquet`, `reports/logs/collection_fastf1.csv`
**Assessment items:** P1 (data collected by the group via web API and web scraping)

| # | Source | Content | Access | Module |
|---|---|---|---|---|
| 1 | FastF1 (F1 live-timing data) | laps, positions, tyres, track status, weather | Python package / web API | `src/collect_fastf1.py` |
| 2 | Jolpica-F1 | calendar, results, qualifying, driver and circuit IDs | REST API (JSON) | `src/collect_jolpica.py` |
| 3 | Wikipedia «List of Formula One circuits» | circuit type, length, turns | web scraping | `src/scrape_wikipedia.py` |
| 4 | bacinger/f1-circuits | circuit layouts and locations (GeoJSON) | file download (pinned commit) | `src/collect_geo.py` |

All collectors first look for the raw files in `data/raw/`. Re-running this notebook therefore does not call the APIs again; `refresh=True` forces a new download. The pilot covers the seasons 2021 (era E1) and 2022 (era E2).

In [1]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [2]:
import json

from src import collect_fastf1, collect_geo, collect_jolpica, scrape_wikipedia
from src.config import PILOT_SEASONS, RAW_DIR

SEASONS = PILOT_SEASONS
SEASONS

(2021, 2022)

## 1. Jolpica-F1 REST API

The API returns at most 100 records per request, so every endpoint is read page by page (`limit`/`offset`). Each page is saved unchanged as JSON; requests are throttled (at least 0.5 s apart) and retried on HTTP 429/5xx.

In [3]:
jolpica = collect_jolpica.collect(SEASONS)
{name: df.shape for name, df in jolpica.items()}

{'races': (44, 10), 'results': (880, 15), 'qualifying': (879, 4)}

In [4]:
# Structure of one raw API page (pagination metadata + nested race table)
page = json.loads((RAW_DIR / "jolpica" / "2021" / "2021_results_offset00000.json").read_text())
mr = page["MRData"]
print({k: mr[k] for k in ("total", "limit", "offset")})
print("first result keys:", list(mr["RaceTable"]["Races"][0]["Results"][0].keys()))

{'total': '440', 'limit': '100', 'offset': '0'}
first result keys: ['number', 'position', 'positionText', 'points', 'Driver', 'Constructor', 'grid', 'laps', 'status', 'Time', 'FastestLap']


In [5]:
jolpica["results"].head()

,season,round,driver_id,code,number,given_name,family_name,nationality,constructor_id,constructor_name,grid,finish_pos,position_text,laps,status
0,2021,1,hamilton,HAM,44,Lewis,Hamilton,British,mercedes,Mercedes,2,1,1,56,Finished
1,2021,1,max_verstappen,VER,33,Max,Verstappen,Dutch,red_bull,Red Bull,1,2,2,56,Finished
2,2021,1,bottas,BOT,77,Valtteri,Bottas,Finnish,mercedes,Mercedes,3,3,3,56,Finished
3,2021,1,norris,NOR,4,Lando,Norris,British,mclaren,McLaren,7,4,4,56,Finished
4,2021,1,perez,PER,11,Sergio,Pérez,Mexican,red_bull,Red Bull,0,5,5,56,Finished


## 2. FastF1 – lap and timing data

FastF1 reads the official F1 live-timing data. Two practical constraints, both handled in `src/collect_fastf1.py`:

- **Access:** since 2026 the live-timing server answers HTTP 401 without an F1 TV account. FastF1 then falls back to its public mirror for historical sessions. The collector never asks for credentials.
- **Rate limit:** FastF1 limits itself to 500 API calls per hour (about 25 races without cache). When the limit is reached, the collector waits and retries instead of skipping races.

Only main races (session `R`) are collected – no sprints, no testing.

In [6]:
ff1_log = collect_fastf1.collect(SEASONS)
ff1_log["status"].value_counts()

2021 R01 Bahrain Grand Prix           cached


2021 R02 Emilia Romagna Grand Prix    cached


2021 R03 Portuguese Grand Prix        cached


2021 R04 Spanish Grand Prix           cached


2021 R05 Monaco Grand Prix            cached


2021 R06 Azerbaijan Grand Prix        cached


... (35 outputs left out)

2022 R20 Mexico City Grand Prix       cached


2022 R21 São Paulo Grand Prix         cached


2022 R22 Abu Dhabi Grand Prix         cached


status
cached    44
Name: count, dtype: int64

In [7]:
ff1_log[["season", "round", "event_name", "event_format", "status",
         "n_laps", "n_weather", "n_track_status", "total_laps"]]

,season,round,event_name,event_format,status,n_laps,n_weather,n_track_status,total_laps
0,2021,1,Bahrain Grand Prix,conventional,cached,1027,133,13,56
1,2021,2,Emilia Romagna Grand Prix,conventional,cached,1127,161,22,63
2,2021,3,Portuguese Grand Prix,conventional,cached,1245,134,3,66
3,2021,4,Spanish Grand Prix,conventional,cached,1247,130,4,66
4,2021,5,Monaco Grand Prix,conventional,cached,1419,135,1,78
5,2021,6,Azerbaijan Grand Prix,conventional,cached,943,172,30,51
6,2021,7,French Grand Prix,conventional,cached,1051,125,1,53
7,2021,8,Styrian Grand Prix,conventional,cached,1296,118,3,71
8,2021,9,Austrian Grand Prix,conventional,cached,1338,156,5,71
9,2021,10,British Grand Prix,sprint,cached,970,184,13,52


In [8]:
# One raw lap table as delivered by FastF1
laps_raw = pd.read_parquet(RAW_DIR / "fastf1" / "2021" / "01_laps.parquet")
print(laps_raw.shape)
laps_raw[["Driver", "LapNumber", "Time", "LapTime", "Position", "Compound",
          "TyreLife", "PitInTime", "PitOutTime", "TrackStatus", "IsAccurate"]].head()

(1027, 31)


,Driver,LapNumber,Time,LapTime,Position,Compound,TyreLife,PitInTime,PitOutTime,TrackStatus,IsAccurate
0,HAM,1.0,0 days 00:39:09.674000,0 days 00:01:59.538000,2.0,MEDIUM,4.0,NaT,NaT,124,False
1,HAM,2.0,0 days 00:41:32.386000,0 days 00:02:22.712000,2.0,MEDIUM,5.0,NaT,NaT,4,False
2,HAM,3.0,0 days 00:44:09.042000,0 days 00:02:36.656000,2.0,MEDIUM,6.0,NaT,NaT,41,False
3,HAM,4.0,0 days 00:45:53.974000,0 days 00:01:44.932000,2.0,MEDIUM,7.0,NaT,NaT,126,False
4,HAM,5.0,0 days 00:47:39.113000,0 days 00:01:45.139000,2.0,MEDIUM,8.0,NaT,NaT,671,False


## 3. Wikipedia – web scraping

The page is downloaded once with an identifying User-Agent and stored unchanged; parsing works on the stored HTML. The circuit table is located by its column names, footnote markers are removed.

In [9]:
print(scrape_wikipedia.download())
wiki = scrape_wikipedia.parse()
print(wiki.shape)
wiki.head()

cached


(78, 11)


,Circuit,Map,Type,Direction,Location,Country,Last length used,Turns,Grands Prix,Season(s),Grands Prix held
0,Adelaide Street Circuit,None,Street circuit,Clockwise,Adelaide,Australia,3.780 km (2.349 mi),16,Australian Grand Prix,1985–1995,11
1,Ain-Diab Circuit,None,Road circuit,Clockwise,Casablanca,Morocco,7.618 km (4.734 mi),18,Moroccan Grand Prix,1958,1
2,Aintree Motor Racing Circuit,None,Road circuit,Clockwise,Aintree,United Kingdom,4.828 km (3.000 mi),12,British Grand Prix,"1955, 1957, 1959, 1961–1962",5
3,Albert Park Circuit *,None,Street circuit,Clockwise,Melbourne,Australia,5.278 km (3.280 mi),16,Australian Grand Prix,"1996–2019, 2022–2026",29
4,Algarve International Circuit†,None,Race circuit,Clockwise,Portimão,Portugal,4.653 km (2.891 mi),15,Portuguese Grand Prix,2020–2021,2


## 4. GeoJSON circuit layouts

The files are taken from a fixed commit of the repository, so every download returns identical data.

In [10]:
print(collect_geo.download())
layouts = collect_geo.load_layouts()
locations = collect_geo.load_locations()
print(layouts.shape, locations.shape, layouts.crs)
layouts.head(3)

{'f1-circuits.geojson': 'cached', 'f1-locations.geojson': 'cached'}


(40, 8) (40, 10) EPSG:4326


,id,Location,Name,opened,firstgp,length,altitude,geometry
0,au-1953,Melbourne,Albert Park Circuit,1953,1996,5278,10,"LINESTRING (144.96864 -37.84976, 144.96763 -37..."
1,bh-2002,Sakhir,Bahrain International Circuit,2002,2004,5412,-16,"LINESTRING (50.51054 26.03177, 50.51063 26.034..."
2,cn-2004,Shanghai,Shanghai International Circuit,2004,2004,5451,14,"LINESTRING (121.21838 31.33684, 121.21741 31.3..."


## 5. Summary

In [11]:
summary = pd.DataFrame([
    ("FastF1", "web API (Python package)", "laps", int(ff1_log["n_laps"].sum())),
    ("FastF1", "web API (Python package)", "weather samples", int(ff1_log["n_weather"].sum())),
    ("Jolpica-F1", "REST API", "races", len(jolpica["races"])),
    ("Jolpica-F1", "REST API", "race results", len(jolpica["results"])),
    ("Jolpica-F1", "REST API", "qualifying results", len(jolpica["qualifying"])),
    ("Wikipedia", "web scraping", "circuits", len(wiki)),
    ("bacinger/f1-circuits", "file download", "circuit layouts", len(layouts)),
], columns=["source", "access", "records", "count"])
summary

,source,access,records,count
0,FastF1,web API (Python package),laps,47333
1,FastF1,web API (Python package),weather samples,7342
2,Jolpica-F1,REST API,races,44
3,Jolpica-F1,REST API,race results,880
4,Jolpica-F1,REST API,qualifying results,879
5,Wikipedia,web scraping,circuits,78
6,bacinger/f1-circuits,file download,circuit layouts,40
